## 🎯 Learning Objectives
* Understand the critical role of evaluation in the lifecycle of advanced AI agents.
* Learn to create and manage robust evaluation datasets using LangSmith.
* Implement continuous integration (CI) evaluations for LangGraph agents to ensure consistent performance.
* Interpret evaluation results from LangSmith to identify agent weaknesses and drive iterative improvements.


## Agent Evaluation with LangSmith Datasets and CI Evals

Building sophisticated multi-agent systems with frameworks like LangGraph is akin to orchestrating a complex symphony. Each agent, a skilled musician, plays its part, but the true measure of success lies in the harmony of the entire ensemble. How do we ensure this harmony? Through rigorous, continuous evaluation.

Traditional software testing, with its focus on deterministic unit and integration tests, falls short for AI agents. Agents exhibit emergent behaviors, interact with dynamic environments, and often produce non-deterministic outputs. We need a more holistic, data-driven approach that can assess an agent's reasoning, tool use, adherence to instructions, and overall effectiveness across a diverse set of scenarios.

This is where **LangSmith** shines as an indispensable platform for agent evaluation. Think of LangSmith as your agent's personal performance coach and quality assurance manager. It provides the infrastructure to:

1.  **Curate Datasets**: Just as a musician practices with specific pieces, agents need to be tested against a representative collection of inputs and expected outcomes. LangSmith allows you to create and manage these "test suites" – called **datasets** – which are collections of input-output pairs, often including ground truth or desired criteria.
2.  **Run Evaluations**: Once you have a dataset, you can run your agent against it. LangSmith executes your agent for each example in the dataset, capturing detailed traces of its execution (LLM calls, tool uses, state changes). This is like recording the musician's performance.
3.  **Apply Evaluators**: After the agent runs, **evaluators** step in. These are functions (either built-in to LangSmith, custom Python functions, or even LLM-based evaluators) that score the agent's output against the ground truth or predefined criteria. They provide objective metrics on performance, correctness, safety, and more. This is like the coach providing detailed feedback on pitch, rhythm, and interpretation.
4.  **Integrate with CI/CD**: The real power comes from integrating this process into your Continuous Integration (CI) pipeline. Every time you push new code or make a change to your agent's prompts or tools, an automated evaluation can run. If the agent's performance drops below a certain threshold, the CI pipeline can fail, preventing regressions and ensuring that only high-quality agents are deployed. This is like having an automated sound engineer who flags any performance that doesn't meet broadcast quality.

By 2026, robust MLOps practices, especially for agentic systems, are non-negotiable. LangSmith's capabilities for dataset management, flexible evaluation, and seamless CI integration are at the forefront of ensuring your advanced LangGraph agents are not just functional, but consistently performant, reliable, and safe in production environments.


In [ ]:
# Ensure you have the necessary libraries installed:
# pip install langchain langgraph langsmith langchain_openai

import os
from typing import List, Dict, Any, Optional
import uuid

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, END

from langsmith import Client
from langsmith.schemas import Run, Example, EvaluationResult, Feedback
from langsmith.evaluation import evaluate

# --- 1. Set up LangSmith Environment Variables ---
# Replace with your actual API key and project name
# It's recommended to set these as environment variables, e.g., in your .bashrc or .zshrc
# export LANGCHAIN_API_KEY="YOUR_LANGCHAIN_API_KEY"
# export LANGCHAIN_TRACING_V2="true"
# export LANGCHAIN_PROJECT="ADV01-L10-Agent-Evals"

os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "YOUR_LANGCHAIN_API_KEY") # Replace if not set
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "ADV01-L10-Agent-Evals") # Default project name
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY") # Replace if not set

# Initialize LangSmith client
client = Client()

print(f"LangSmith Project: {os.environ['LANGCHAIN_PROJECT']}")

# --- 2. Define a Simple LangGraph Agent ---
# This agent will act as a 'Research Assistant' that can use a dummy search tool.

@tool
def dummy_search(query: str) -> str:
    """Searches a dummy knowledge base for information based on the query."""
    if "LangGraph" in query:
        return "LangGraph is a library for building robust, stateful, multi-actor applications with LLMs."
    elif "AI Agent" in query:
        return "An AI agent is an autonomous entity that perceives its environment and takes actions to achieve goals."
    elif "evaluation" in query:
        return "Evaluation of AI agents involves assessing their performance, reliability, and safety across various metrics."
    else:
        return f"No specific information found for '{query}'."

# Define the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Define the agent state
class AgentState(Dict):
    messages: List[BaseMessage]
    tool_calls: List[Dict] # To store tool calls for the agent to process
    tool_output: Optional[str] = None # To store the output of the tool

# Define the nodes
def call_llm(state: AgentState) -> AgentState:
    messages = state["messages"]
    response = llm.invoke(messages)
    # Check if the LLM wants to call a tool
    if response.tool_calls:
        return {"messages": messages + [response], "tool_calls": response.tool_calls}
    else:
        return {"messages": messages + [response]}

def call_tool(state: AgentState) -> AgentState:
    tool_calls = state["tool_calls"]
    tool_output_messages = []
    for tool_call in tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        if tool_name == "dummy_search":
            output = dummy_search.invoke(tool_args)
            tool_output_messages.append(AIMessage(content=f"Tool output for {tool_name}: {output}"))
            state["tool_output"] = output # Store for potential LLM processing
    return {"messages": state["messages"] + tool_output_messages}

# Build the graph
workflow = StateGraph(AgentState)

workflow.add_node("llm", call_llm)
workflow.add_node("tool", call_tool)

workflow.set_entry_point("llm")

workflow.add_conditional_edges(
    "llm",
    lambda state: "tool" if state.get("tool_calls") else END,
    {"tool": "tool", END: END}
)
workflow.add_edge("tool", "llm") # After tool call, go back to LLM to summarize/respond

app = workflow.compile()

# --- 3. Create a LangSmith Dataset ---

dataset_name = f"Research Agent Evaluation Dataset - {uuid.uuid4().hex[:6]}"

# Check if dataset already exists to avoid duplicates in a real CI scenario
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Using existing dataset: {dataset_name}")
except:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="Dataset for evaluating a simple research assistant agent."
    )
    print(f"Created new dataset: {dataset_name}")

# Add examples to the dataset
examples_to_add = [
    {
        "input": {"messages": [HumanMessage(content="What is LangGraph?")]},
        "ground_truth": "LangGraph is a library for building robust, stateful, multi-actor applications with LLMs."
    },
    {
        "input": {"messages": [HumanMessage(content="Explain what an AI Agent is.")]},
        "ground_truth": "An AI agent is an autonomous entity that perceives its environment and takes actions to achieve goals."
    },
    {
        "input": {"messages": [HumanMessage(content="How is AI agent evaluation typically done?")]},
        "ground_truth": "Evaluation of AI agents involves assessing their performance, reliability, and safety across various metrics."
    },
    {
        "input": {"messages": [HumanMessage(content="Tell me about the capital of France.")]},
        "ground_truth": "No specific information found for 'capital of France'."
    }
]

# Clear existing examples if re-running, then add new ones
# In a real scenario, you'd manage dataset updates more carefully.
existing_examples = client.list_examples(dataset_id=dataset.id)
for ex in existing_examples:
    client.delete_example(example_id=ex.id)

for ex_data in examples_to_add:
    client.create_example(
        dataset_id=dataset.id,
        inputs=ex_data["input"],
        outputs=None, # Outputs are generated by the agent during evaluation
        ground_truth=ex_data["ground_truth"]
    )
print(f"Added {len(examples_to_add)} examples to dataset '{dataset_name}'.")

# --- 4. Define Custom Evaluators ---

# Custom Python evaluator: Checks if a specific keyword is present in the final output.
# This is a simple heuristic, but demonstrates the structure.
def keyword_presence_evaluator(run: Run, example: Example) -> EvaluationResult:
    agent_output = run.outputs.get("messages")[-1].content if run.outputs and run.outputs.get("messages") else ""
    ground_truth = example.ground_truth

    # Simple check: Does the agent's final response contain a key part of the ground truth?
    # This is very basic; real evaluators would be more sophisticated.
    is_correct = ground_truth in agent_output or (ground_truth == "No specific information found for 'capital of France'." and "No specific information found" in agent_output)

    score = 1.0 if is_correct else 0.0
    feedback_key = "keyword_match"
    comment = f"Agent output: '{agent_output}', Ground truth: '{ground_truth}'"

    return EvaluationResult(
        key=feedback_key,
        score=score,
        comment=comment,
        correction=None, # Can be used for suggested corrections
        evaluator_info={
            "evaluator_type": "custom_python",
            "criteria": "Checks if ground truth is present in agent's final output."
        }
    )

# LangSmith's built-in CriteriaEvaluator (LLM-based)
# This uses an LLM to judge the output based on predefined criteria.
# Ensure your OPENAI_API_KEY is set for this to work.
from langsmith.evaluation import CriteriaEvaluator

llm_criteria_evaluator = CriteriaEvaluator(
    llm=llm,
    criteria={
        "correctness": "Is the agent's final answer factually correct and directly addresses the user's query?",
        "completeness": "Does the agent's answer provide a comprehensive response based on the available information?",
        "tool_use_efficiency": "Did the agent use the dummy_search tool appropriately and efficiently?"
    },
    # You can also provide examples for few-shot evaluation
    # examples=[
    #     {"input": "What is LangGraph?", "output": "LangGraph is a framework.", "criteria": {"correctness": "No"}}
    # ]
)

# --- 5. Run Evaluation on the Dataset ---

print(f"Running evaluation on dataset '{dataset_name}'...")

# The 'evaluate' function runs the agent against the dataset and applies evaluators.
# It returns a dictionary with evaluation results and a link to the LangSmith run.

eval_results = evaluate(
    run_on_dataset=app, # The LangGraph compiled app
    data=dataset_name, # The name of the dataset to use
    evaluators=[
        keyword_presence_evaluator, # Our custom Python evaluator
        llm_criteria_evaluator      # LangSmith's built-in LLM-based evaluator
    ],
    experiment_prefix="CI-Eval-Agent-V1", # Prefix for the experiment name in LangSmith
    metadata={
        "commit_hash": "abc123def456", # Example of CI metadata
        "branch": "main"
    },
    max_concurrency=5 # Adjust based on API rate limits and desired speed
)

print("Evaluation complete!")
print(f"View results in LangSmith: {eval_results['url']}")

# --- 6. Simulate CI Check (Conceptual) ---
# In a real CI pipeline, you would fetch these results and apply pass/fail logic.

# Example: Check average score for 'keyword_match' evaluator
keyword_scores = [res.score for res in eval_results["results"].values() if res.get("feedback") and res["feedback"].get("keyword_match")]
if keyword_scores:
    avg_keyword_score = sum(keyword_scores) / len(keyword_scores)
    print(f"Average 'keyword_match' score: {avg_keyword_score:.2f}")
    if avg_keyword_score < 0.75: # Define your threshold
        print("CI FAILED: Agent performance for keyword matching is below threshold!")
        # In a real CI, this would be a sys.exit(1) or similar.
    else:
        print("CI PASSED: Agent performance for keyword matching is satisfactory.")

# Example: Check LLM-based correctness score
correctness_scores = []
for res in eval_results["results"].values():
    if res.get("feedback") and res["feedback"].get("correctness"):
        # LLM-based evaluators often return 'Y' or 'N' for criteria
        score = 1.0 if res["feedback"]["correctness"].score == 'Y' else 0.0
        correctness_scores.append(score)

if correctness_scores:
    avg_correctness_score = sum(correctness_scores) / len(correctness_scores)
    print(f"Average LLM-based 'correctness' score: {avg_correctness_score:.2f}")
    if avg_correctness_score < 0.8: # Define your threshold
        print("CI FAILED: Agent correctness is below threshold!")
    else:
        print("CI PASSED: Agent correctness is satisfactory.")

print("\n--- End of Evaluation Script ---")


### Interpreting Evaluation Output and Practical Considerations

After running the code, you'll see output indicating the evaluation has completed and, most importantly, a URL to the LangSmith experiment. This URL is your gateway to understanding your agent's performance in detail.

#### Interpreting LangSmith Results:

1.  **Experiment View**: In LangSmith, navigate to the provided URL. You'll see an "Experiment" dashboard. Each row represents a run of your agent against an example from your dataset. You can compare different experiments (e.g., `CI-Eval-Agent-V1` vs. `CI-Eval-Agent-V2` after a code change).
2.  **Trace Details**: Click on any individual run to see its full trace. This is incredibly powerful for debugging and understanding agent behavior:
    *   **LLM Calls**: See the exact prompts sent to the LLM and its responses.
    *   **Tool Calls**: Observe which tools were invoked, with what arguments, and their outputs.
    *   **State Changes**: For LangGraph, you can often see the state transitions between nodes.
    *   **Evaluator Feedback**: Each evaluator you defined (e.g., `keyword_match`, `correctness`, `completeness`) will have its own section, showing its score and any comments. For LLM-based evaluators, you'll even see the LLM's reasoning for its judgment.
3.  **Aggregate Metrics**: The experiment view provides aggregate statistics (average scores, pass rates) across all examples for each evaluator. This gives you a high-level overview of your agent's performance.

#### Performance Trade-offs and Best Practices:

*   **Cost and Latency**: Running evaluations, especially with LLM-based evaluators, incurs API costs and takes time. For CI, you might start with a smaller, critical dataset and expand to a larger, more comprehensive one for nightly or weekly runs.
*   **Dataset Quality**: The quality of your evaluation is directly tied to the quality and diversity of your dataset. Include a mix of common cases, edge cases, and known failure modes. Continuously update your dataset with real-world examples that cause your agent to fail.
*   **Evaluator Selection**: 
    *   **Heuristic/Rule-based**: Fast and cheap, good for specific, deterministic checks (e.g., keyword presence, JSON format validation).
    *   **LLM-based**: More flexible and powerful for nuanced judgments (e.g., correctness, coherence, tone), but more expensive and slower. Use them for critical, subjective aspects.
    *   **Human-in-the-Loop**: For the most critical applications, human review and feedback remain invaluable. LangSmith supports collecting human feedback directly.
*   **Thresholds for CI**: Set realistic and actionable performance thresholds. If an agent's average correctness score drops from 90% to 70%, that's a clear signal to investigate before deployment.
*   **Iterative Improvement**: Evaluation is not a one-time event. It's a continuous loop: Evaluate -> Analyze -> Improve Agent -> Re-evaluate. Use the insights from LangSmith traces to refine prompts, adjust tool use logic, or modify graph structures.

#### Typical Use Cases:

*   **Regression Testing**: Ensure new code changes, prompt updates, or model versions don't degrade existing performance.
*   **A/B Testing**: Compare two different agent architectures or prompt strategies against the same dataset to determine which performs better.
*   **Production Monitoring**: Continuously evaluate agent performance in production by sampling real user interactions and adding them to your evaluation dataset.
*   **Quality Gates**: Automate pass/fail decisions in your CI/CD pipeline, preventing underperforming agents from reaching production.
*   **Model Selection**: Evaluate different underlying LLMs (e.g., GPT-4o vs. Claude 3.5 Sonnet) for your agent's specific tasks.

By embedding LangSmith evaluations into your development workflow, you transform agent development from a trial-and-error process into a data-driven, quality-controlled engineering discipline, essential for building robust and reliable advanced AI agents in 2026 and beyond.


### Resources

*   **LangSmith Documentation - Datasets**: [https://docs.smith.langchain.com/concepts/datasets](https://docs.smith.langchain.com/concepts/datasets)
*   **LangSmith Documentation - Evaluators**: [https://docs.smith.langchain.com/concepts/evaluation/evaluators](https://docs.smith.langchain.com/concepts/evaluation/evaluators)
*   **LangSmith Documentation - CI/CD Integration**: [https://docs.smith.langchain.com/how_to_guides/ci_cd](https://docs.smith.langchain.com/how_to_guides/ci_cd)
*   **LangGraph Documentation**: [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain Blog - Agent Evaluation**: [https://blog.langchain.dev/agent-evaluation/](https://blog.langchain.dev/agent-evaluation/) (While slightly older, the core concepts remain highly relevant.)
*   **OpenAI API Documentation**: [https://platform.openai.com/docs/api-reference](https://platform.openai.com/docs/api-reference)
*   **General MLOps Best Practices**: Explore resources from Google Cloud AI, AWS SageMaker, or Microsoft Azure ML for broader MLOps context.
